# LeetCode 2879 — Display the First Three Rows bằng Python (pandas)

Link đề: https://leetcode.com/problems/display-the-first-three-rows/ · Độ khó: **Easy** · Chủ đề: Pandas (30 Days of Pandas)

Skeleton LeetCode (không có `class Solution`; tên hàm / chữ ký: đối chiếu lại với đề gốc):

```python
import pandas as pd

def selectFirstRows(employees: pd.DataFrame) -> pd.DataFrame:
    ...
```

Ghi chú: dữ liệu ví dụ trong notebook được viết theo trí nhớ về đề; nên đối chiếu lại với đề gốc nếu muốn khớp từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trả về một DataFrame chứa **3 hàng đầu tiên** của `employees` (giữ nguyên các cột).
2. Biết các cách lấy "vài hàng đầu" trong pandas: `head(n)`, cắt lát theo vị trí `iloc[:n]`, cắt lát `df[:n]`, chọn theo nhãn `loc[df.index[:n]]`, và **vòng lặp thuần Python** để làm baseline.
3. Phân biệt **chọn theo vị trí (`iloc`)** với **chọn theo nhãn (`loc`)**; hiểu vì sao `loc[:2]` lấy **3 hàng** (cận cuối **được gồm**) còn `iloc[:3]` cũng 3 hàng nhưng cận cuối **không gồm**.
4. Biết kết quả giữ **index gốc** (`0, 1, 2`), **cột và dtype** không đổi; và hiểu rằng đây là một **bản mới** (không sửa `employees`).
5. Tránh các lỗi kinh điển: `head(-3)`, `loc[:3]` lấy 4 hàng, nhầm `df[3]` với cắt hàng, quên `return` (chỉ `print`), nhầm `head()` mặc định là 5 hàng.

## 2. Đề bài & ví dụ

Cho một DataFrame `employees` có các cột `employee_id`, `name`, `department`, `salary`.

Hãy viết hàm hiển thị (trả về) **3 hàng đầu tiên** của DataFrame đó.

**Ví dụ**

- Input: DataFrame `employees`

| employee_id | name | department | salary |
|---|---|---|---|
| 3 | Bob | Operations | 48675 |
| 90 | Alice | Sales | 11096 |
| 9 | Tatiana | Engineering | 33805 |
| 60 | Annabelle | InfoTech | 37678 |
| 49 | Jonathan | Engineering | 23678 |
| 43 | Khaled | Engineering | 78484 |

- Output:

| employee_id | name | department | salary |
|---|---|---|---|
| 3 | Bob | Operations | 48675 |
| 90 | Alice | Sales | 11096 |
| 9 | Tatiana | Engineering | 33805 |

- Giải thích: chỉ giữ 3 hàng đầu tiên (các hàng ở vị trí 0, 1, 2); tất cả cột giữ nguyên.

(Giá trị cụ thể trong bảng: đối chiếu lại với đề gốc.)

In [1]:
from typing import List
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    "employee_id": [3, 90, 9, 60, 49, 43],
    "name": ["Bob", "Alice", "Tatiana", "Annabelle", "Jonathan", "Khaled"],
    "department": ["Operations", "Sales", "Engineering", "InfoTech", "Engineering", "Engineering"],
    "salary": [48675, 11096, 33805, 37678, 23678, 78484],
})
print(employees)
print("shape:", employees.shape)

   employee_id       name   department  salary
0            3        Bob   Operations   48675
1           90      Alice        Sales   11096
2            9    Tatiana  Engineering   33805
3           60  Annabelle     InfoTech   37678
4           49   Jonathan  Engineering   23678
5           43     Khaled  Engineering   78484
shape: (6, 4)


## 3. Kiến thức nền

### 3.1. Index: nhãn của hàng

Mỗi hàng của DataFrame có một **nhãn** gọi là **index**. Mặc định là `0, 1, 2, …`. Nhãn **khác** với **vị trí**: vị trí là số thứ tự thật (0 là hàng đầu tiên) còn nhãn có thể bị đổi (ví dụ sau khi lọc hoặc sắp xếp).

In [2]:
print(employees.index)                       # RangeIndex(start=0, stop=6, step=1)
filtered = employees[employees["salary"] > 30000]
print(filtered)                              # nhãn còn lại: 0, 2, 3, 5 (vị trí: 0, 1, 2, 3)
print(filtered.index.tolist())

RangeIndex(start=0, stop=6, step=1)
   employee_id       name   department  salary
0            3        Bob   Operations   48675
2            9    Tatiana  Engineering   33805
3           60  Annabelle     InfoTech   37678
5           43     Khaled  Engineering   78484
[0, 2, 3, 5]


### 3.2. `df.head(n)`

`df.head(n)` trả về **n hàng đầu tiên theo vị trí**. Mặc định `n = 5`. Nếu bảng có ít hơn `n` hàng, trả về tất cả.

In [3]:
print(employees.head(3))
print(len(employees.head()))        # 5 (mặc định)
print(len(employees.head(100)))     # 6 (nhiều hơn số hàng -> lấy hết)

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805
5
6


### 3.3. Cắt lát theo vị trí: `df.iloc[a:b]`

`iloc` = **i**nteger **loc**ation: chỉ định bằng **vị trí số nguyên**. Quy tắc cắt lát giống list: `a` được gồm, `b` **không** được gồm. `df.iloc[:3]` = các vị trí `0, 1, 2`.

In [4]:
print(employees.iloc[:3])
print(employees.iloc[1:3])       # vị trí 1 và 2

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805
   employee_id     name   department  salary
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


### 3.4. Cắt lát ngắn gọn `df[a:b]`

Khi dùng **một lát cắt** (có dấu `:`) trong `df[...]`, pandas hiểu là **cắt hàng theo vị trí**. Nhưng `df["name"]` (một chuỗi) lại là **chọn cột**. Hai nghĩa khác nhau nên dễ gây nhầm.

In [5]:
print(employees[:3])             # 3 hàng đầu (cắt hàng)
print(employees["name"].tolist())  # chọn cột "name"

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805
['Bob', 'Alice', 'Tatiana', 'Annabelle', 'Jonathan', 'Khaled']


### 3.5. Chọn theo nhãn: `df.loc[...]`

`loc` chọn theo **nhãn**. Với lát cắt `df.loc[a:b]`, cả `a` và `b` **đều được gồm** (khác `iloc`). Vì thế `loc[:2]` lấy nhãn `0, 1, 2` (3 hàng). Điều này chỉ đúng khi nhãn là `0, 1, 2, …`.

In [6]:
print(employees.loc[:2])         # nhãn 0, 1, 2 -> 3 hàng (cận cuối được gồm)
print(employees.loc[employees.index[:3]])     # lấy nhãn của 3 hàng đầu rồi chọn theo nhãn

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805
   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


### 3.4b. Bản sao hay bản gốc? (Copy-on-Write)

Từ pandas 3.0, mọi kết quả chọn / cắt lát đều **hành xử như một bản sao độc lập**: sửa kết quả **không** làm đổi `employees` (pandas sao chép "lười", chỉ khi cần). Vì vậy hàm của ta trả về `head(3)` mà không lo ảnh hưởng đến dữ liệu gốc.

In [7]:
h = employees.head(3)
h.loc[0, "salary"] = 0
print(h["salary"].tolist())                 # [0, 11096, 33805]
print(employees["salary"].tolist()[:3])     # [48675, 11096, 33805]  (bảng gốc không đổi)

[0, 11096, 33805]
[48675, 11096, 33805]


### 3.6. `zip`, `enumerate` và `break` (cho vòng lặp thuần)

`for pos, (label, row) in enumerate(zip(df.index, df.itertuples(index=False, name=None)))` cho ta vị trí `pos`, nhãn `label` và hàng `row` (một tuple giá trị). Có `break` khi `pos == 3` để dừng sớm.

In [8]:
for pos, (label, row) in enumerate(zip(employees.index, employees.itertuples(index=False, name=None))):
    print(pos, label, row)
    if pos == 2:
        break

0 0 (3, 'Bob', 'Operations', 48675)
1 1 (90, 'Alice', 'Sales', 11096)
2 2 (9, 'Tatiana', 'Engineering', 33805)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hình dung bảng như một chồng giấy. "3 hàng đầu" = **ba tờ trên cùng**, cột giữ nguyên:

```
vị trí 0 → Bob        ✓ lấy
vị trí 1 → Alice      ✓ lấy
vị trí 2 → Tatiana    ✓ lấy
vị trí 3 → Annabelle  ✗ bỏ
vị trí 4 → Jonathan   ✗ bỏ
vị trí 5 → Khaled     ✗ bỏ
```

Kết quả: 3 hàng, vẫn đủ 4 cột, **nhãn hàng giữ nguyên `0, 1, 2`** (không đánh số lại vì đó cũng là ba nhãn đầu).

Nếu bảng chỉ có 2 hàng thì "3 hàng đầu" chính là cả 2 hàng. Ta không muốn code báo lỗi trong trường hợp này.

## 5. Cách 1 — Brute Force / baseline: vòng lặp thuần Python

Duyệt từng hàng, gom 3 hàng đầu vào list, **dừng sớm** bằng `break`, rồi dựng lại DataFrame. Ta giữ nhãn (`index`) gốc và tên cột để kết quả y hệt.

In [9]:
def first_rows_loop(employees: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    labels, rows = [], []
    for pos, (label, row) in enumerate(zip(employees.index, employees.itertuples(index=False, name=None))):
        if pos == n:
            break
        labels.append(label)
        rows.append(row)
    return pd.DataFrame(rows, index=labels, columns=employees.columns)

out = first_rows_loop(employees)
print(out)
print(out.dtypes.tolist())

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805
[dtype('int64'), <StringDtype(storage='python', na_value=nan)>, <StringDtype(storage='python', na_value=nan)>, dtype('int64')]


**Độ phức tạp:** duyệt tối đa `n = 3` hàng (hằng số) nhưng mỗi hàng có `c` cột nên **Time `O(c)`** (xem `n` là hằng số), **Space `O(c)`**.

**Nút thắt:** code dài, phải tự lo nhãn / tên cột / dtype (nếu bảng gốc rỗng, dtype sẽ khác). pandas có sẵn phép "lấy n hàng đầu" làm đúng tất cả những điều đó.

## 6. Ý tưởng thuật toán / bước đột phá

"Lấy `n` hàng đầu" là thao tác **cắt lát theo vị trí** trên chiều hàng: giữ vị trí `0 … n-1`, giữ nguyên mọi cột. pandas cung cấp trực tiếp:

- `df.head(n)` — tên rõ nghĩa nhất, tự xử lý khi `n` lớn hơn số hàng;
- `df.iloc[:n]` — cắt lát theo vị trí, tương đương;
- `df[:n]` — cú pháp tắt.

Cả ba **không duyệt từng hàng bằng Python**; pandas chỉ cần ghi nhớ "lấy các vị trí 0..n-1" nên rất nhanh.

## 7. Các cách tiếp theo

### Cách 2 — `df.head(3)` (cách chuẩn)

**Giải thích.** `head(3)` trả về DataFrame mới chứa 3 hàng đầu, cùng cột, cùng dtype, cùng nhãn.

In [10]:
def first_rows_head(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.head(3)

print(first_rows_head(employees))

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


In [11]:
# Code nộp LeetCode cho Cách 2
import pandas as pd

def selectFirstRows(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.head(3)

print(selectFirstRows(employees))

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


**Độ phức tạp Cách 2:** **Time `O(1)` đến `O(c)`** (không phụ thuộc số hàng của bảng gốc), **Space `O(c)`** cho 3 hàng kết quả.

---

### Cách 3 — Cắt lát theo vị trí `iloc[:3]`

**Giải thích.** `iloc[:3]` lấy vị trí `0, 1, 2`. Giống list: cận cuối **không** gồm. Rất hợp khi muốn nhấn mạnh "lấy theo **vị trí**".

In [12]:
def first_rows_iloc(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.iloc[:3]

print(first_rows_iloc(employees))

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


In [13]:
# Code nộp LeetCode cho Cách 3
import pandas as pd

def selectFirstRows(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.iloc[:3]

print(selectFirstRows(employees))

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


**Độ phức tạp Cách 3:** như Cách 2.

---

### Cách 4 — Cắt lát tắt `df[:3]`

**Giải thích.** `employees[:3]` cắt hàng theo vị trí (vì có dấu `:`). Ngắn nhất, nhưng **dễ gây hiểu nhầm** với `df["tên cột"]`, nên đồng đội đọc code có thể phải suy nghĩ thêm.

In [14]:
def first_rows_slice(employees: pd.DataFrame) -> pd.DataFrame:
    return employees[:3]

print(first_rows_slice(employees))

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


**Độ phức tạp Cách 4:** như Cách 2.

---

### Cách 5 — Chọn theo nhãn `loc[df.index[:3]]` hoặc `take`

**Giải thích.**

- `employees.index[:3]` lấy **3 nhãn đầu**; `employees.loc[nhãn]` chọn các hàng có nhãn đó. Cách này **luôn đúng** kể cả khi index không phải `0, 1, 2, …`.
- `employees.take([0, 1, 2])` chọn theo vị trí; nhưng báo lỗi nếu bảng có ít hơn 3 hàng, nên cần `range(min(3, len(df)))`.

In [15]:
def first_rows_loc(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.loc[employees.index[:3]]

def first_rows_take(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.take(list(range(min(3, len(employees)))))

print(first_rows_loc(employees))
print(first_rows_take(employees))

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805
   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


**Độ phức tạp Cách 5:** **Time `O(1)` đến `O(c)`**; lưu ý `loc` với index **bị trùng nhãn** có thể trả nhiều hơn 3 hàng (xem Lỗi 5 ở mục 13).

## 8. TRACE từng bước

Ta in DataFrame ở **mỗi bước** khi mở rộng dần "n hàng đầu", rồi xem `head(n)` với nhiều giá trị `n` khác nhau (kể cả bất thường).

In [16]:
def trace_grow(employees):
    print("Bảng gốc:", employees.shape)
    for n in range(1, 5):
        part = employees.iloc[:n]
        print(f"\n--- iloc[:{n}] -> shape {part.shape}, nhãn {part.index.tolist()}")
        print(part)

trace_grow(employees)

Bảng gốc: (6, 4)

--- iloc[:1] -> shape (1, 4), nhãn [0]
   employee_id name  department  salary
0            3  Bob  Operations   48675

--- iloc[:2] -> shape (2, 4), nhãn [0, 1]
   employee_id   name  department  salary
0            3    Bob  Operations   48675
1           90  Alice       Sales   11096

--- iloc[:3] -> shape (3, 4), nhãn [0, 1, 2]
   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805

--- iloc[:4] -> shape (4, 4), nhãn [0, 1, 2, 3]
   employee_id       name   department  salary
0            3        Bob   Operations   48675
1           90      Alice        Sales   11096
2            9    Tatiana  Engineering   33805
3           60  Annabelle     InfoTech   37678


In [17]:
for n in [0, 1, 3, 6, 100, -2]:
    part = employees.head(n)
    print(f"head({n:>3}) -> {len(part)} hàng, nhãn {part.index.tolist()}")

head(  0) -> 0 hàng, nhãn []
head(  1) -> 1 hàng, nhãn [0]
head(  3) -> 3 hàng, nhãn [0, 1, 2]
head(  6) -> 6 hàng, nhãn [0, 1, 2, 3, 4, 5]
head(100) -> 6 hàng, nhãn [0, 1, 2, 3, 4, 5]
head( -2) -> 4 hàng, nhãn [0, 1, 2, 3]


Quan sát: `head(0)` rỗng; `head(100)` lấy tất cả 6 hàng; `head(-2)` lấy **tất cả trừ 2 hàng cuối** (4 hàng) — đây là hành vi của số âm.

Trace vòng lặp thuần (Cách 1):

In [18]:
def trace_loop(employees, n=3):
    labels, rows = [], []
    for pos, (label, row) in enumerate(zip(employees.index, employees.itertuples(index=False, name=None))):
        if pos == n:
            print(f"pos={pos}: đủ {n} hàng -> break")
            break
        labels.append(label)
        rows.append(row)
        print(f"pos={pos}: thêm nhãn {label}, hàng {row}")
    out = pd.DataFrame(rows, index=labels, columns=employees.columns)
    print(out)
    return out

_ = trace_loop(employees)

pos=0: thêm nhãn 0, hàng (3, 'Bob', 'Operations', 48675)
pos=1: thêm nhãn 1, hàng (90, 'Alice', 'Sales', 11096)
pos=2: thêm nhãn 2, hàng (9, 'Tatiana', 'Engineering', 33805)
pos=3: đủ 3 hàng -> break
   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


## 9. Code nộp LeetCode

In [19]:
import pandas as pd

def selectFirstRows(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.head(3)

result = selectFirstRows(employees)
print(result)
result

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805


,employee_id,name,department,salary
0,3,Bob,Operations,48675
1,90,Alice,Sales,11096
2,9,Tatiana,Engineering,33805


## 10. Vì sao thuật toán đúng

- **Đúng 3 hàng đầu:** `head(3)` giữ các vị trí `0, 1, 2` (theo thứ tự hiện có của bảng), tức đúng "3 hàng đầu tiên".
- **Thứ tự được giữ:** không có bước sắp xếp nên thứ tự hàng không đổi.
- **Cột giữ nguyên:** `head` chỉ lọc trên chiều hàng; tên, thứ tự và dtype các cột không đổi.
- **Index giữ nguyên:** nhãn của các hàng được lấy được giữ lại (ở ví dụ là `0, 1, 2`).
- **Không lỗi khi ít hơn 3 hàng:** `head` trả về toàn bộ bảng; điều này khớp ý "tối đa 3 hàng".
- **Không sửa đầu vào:** kết quả là một bảng mới (Copy-on-Write), nên `employees` không bị thay đổi.

## 11. Phân tích Time / Space Complexity

Gọi `m` là số hàng của bảng gốc, `c` là số cột, `k = 3`.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp thuần Python | `O(k·c)` | `O(k·c)` |
| 2. `head(3)` | `O(k·c)` (không phụ thuộc `m`) | `O(k·c)` |
| 3. `iloc[:3]` | `O(k·c)` | `O(k·c)` |
| 4. `df[:3]` | `O(k·c)` | `O(k·c)` |
| 5. `loc[index[:3]]` / `take` | `O(k·c)` | `O(k·c)` |

Với `k = 3` cố định, mọi cách đều gần như hằng số. Cái khác biệt thật sự là **độ an toàn** (index trùng, bảng ngắn) và **độ rõ ràng** của code.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

So từng cách với DataFrame mong đợi (dựng tay) bằng `pd.testing.assert_frame_equal` (kiểm cột, dtype, index).

In [20]:
impls = {
    "loop": first_rows_loop,
    "head": first_rows_head,
    "iloc": first_rows_iloc,
    "slice": first_rows_slice,
    "loc": first_rows_loc,
    "take": first_rows_take,
    "submit": selectFirstRows,
}

expected_example = pd.DataFrame({
    "employee_id": [3, 90, 9],
    "name": ["Bob", "Alice", "Tatiana"],
    "department": ["Operations", "Sales", "Engineering"],
    "salary": [48675, 11096, 33805],
})
for key, fn in impls.items():
    pd.testing.assert_frame_equal(fn(employees), expected_example)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


In [21]:
two_rows = employees.iloc[:2]
exactly3 = employees.iloc[:3]
one_row = employees.iloc[:1]
for name, df in [("2 hàng", two_rows), ("đúng 3 hàng", exactly3), ("1 hàng", one_row)]:
    for key, fn in impls.items():
        got = fn(df)
        assert got.shape == (min(3, len(df)), df.shape[1]), (name, key, got.shape)
        pd.testing.assert_frame_equal(got, df.iloc[:3])
    print("PASS", name, "-> trả", min(3, len(df)), "hàng")

# bảng rỗng: dtype có thể khác ở baseline nên chỉ kiểm hình dạng và tên cột
empty = employees.iloc[:0]
for key, fn in impls.items():
    got = fn(empty)
    assert got.shape == (0, 4) and list(got.columns) == list(employees.columns), key
print("PASS bảng rỗng")

PASS 2 hàng -> trả 2 hàng
PASS đúng 3 hàng -> trả 3 hàng
PASS 1 hàng -> trả 1 hàng
PASS bảng rỗng


### 12.2. Index không phải `0..n-1` và phản ví dụ

Sau khi **sắp xếp / lọc**, nhãn không còn là `0, 1, 2…`. Đây là chỗ mà `loc[:2]` và `iloc[:3]` cho kết quả **khác nhau**.

In [22]:
shuffled = employees.sort_values("salary", ascending=False)       # nhãn bị xáo trộn
print(shuffled.index.tolist())                                    # [5, 0, 3, 2, 4, 1]
for key, fn in impls.items():
    got = fn(shuffled)
    assert got.index.tolist() == shuffled.index.tolist()[:3], key   # giữ nhãn gốc
    pd.testing.assert_frame_equal(got, shuffled.iloc[:3])
print("PASS bảng đã sắp xếp: mọi cách lấy đúng 3 hàng đầu theo VỊ TRÍ")

# phản ví dụ: loc[:2] chọn theo NHÃN, không phải vị trí
wrong = shuffled.loc[:2]
print("loc[:2] ->", wrong.index.tolist(), "(dừng ở nhãn 2 chứ không phải vị trí 2)")
assert wrong.index.tolist() != shuffled.index.tolist()[:3]

[5, 0, 3, 2, 4, 1]
PASS bảng đã sắp xếp: mọi cách lấy đúng 3 hàng đầu theo VỊ TRÍ
loc[:2] -> [5, 0, 3, 2] (dừng ở nhãn 2 chứ không phải vị trí 2)


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh bảng ngẫu nhiên có `m ∈ [0, 30]` hàng, kèm index ngẫu nhiên **không trùng**, so mọi cách với `df.iloc[:3]`.

In [23]:
import random
random.seed(2879)
np.random.seed(2879)
for t in range(200):
    m = random.randint(1, 30)
    df = pd.DataFrame({
        "employee_id": np.random.randint(1, 1000, size=m),
        "name": [random.choice(["Ann", "Bao", "Chi", "Dung"]) for _ in range(m)],
        "salary": np.random.randint(1000, 90000, size=m),
    })
    if random.random() < 0.5:
        df.index = random.sample(range(1000), m)             # index lộn xộn, không trùng
    exp = df.iloc[:3]
    for key, fn in impls.items():
        got = fn(df)
        pd.testing.assert_frame_equal(got, exp)
        assert len(got) == min(3, m)
    before = df.copy()
    selectFirstRows(df)
    pd.testing.assert_frame_equal(df, before)                # đầu vào không bị sửa
print("200 bảng ngẫu nhiên: mọi cách khớp iloc[:3], đầu vào không đổi")

200 bảng ngẫu nhiên: mọi cách khớp iloc[:3], đầu vào không đổi


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `head(-3)` — số âm không phải "ba hàng đầu"

In [24]:
def sai_head_am(employees):
    return employees.head(-3)             # sai

print("sai :", len(sai_head_am(employees)), "hàng ->", sai_head_am(employees)["name"].tolist())
print("đúng:", len(selectFirstRows(employees)), "hàng ->", selectFirstRows(employees)["name"].tolist())

sai : 3 hàng -> ['Bob', 'Alice', 'Tatiana']
đúng: 3 hàng -> ['Bob', 'Alice', 'Tatiana']


**Vì sao sai:** số âm trong `head(-k)` nghĩa là "**bỏ `k` hàng cuối**", trả về `m - k` hàng. Muốn ba hàng đầu dùng `head(3)`.

### Lỗi 2: `loc[:3]` lấy 4 hàng

In [25]:
def sai_loc(employees):
    return employees.loc[:3]               # sai: nhãn 0,1,2,3 (cận cuối được gồm)

print("sai :", len(sai_loc(employees)), "hàng")
print("đúng:", len(selectFirstRows(employees)), "hàng")

sai : 4 hàng
đúng: 3 hàng


**Vì sao sai:** lát cắt theo **nhãn** (`loc`) **gồm** cả cận cuối. Với lát cắt theo vị trí dùng `iloc[:3]` (không gồm cận cuối) hoặc `head(3)`.

### Lỗi 3: Nhầm `df[3]` / `df[:3]`

In [26]:
try:
    employees[3]                           # sai: ý ta là hàng, nhưng đây là chọn CỘT tên 3
except KeyError as e:
    print("KeyError:", e)
print(len(employees[:3]), "hàng với lát cắt df[:3]")

KeyError: 3
3 hàng với lát cắt df[:3]


**Vì sao sai:** `df[3]` (một số hoặc chuỗi, không có dấu `:`) là chọn **cột** có tên `3`. Muốn hàng đầu, dùng `df.iloc[3]` hoặc lát cắt `df[:3]`.

### Lỗi 4: Chỉ `print` mà không `return`

In [27]:
def sai_chi_in(employees):
    print(employees.head(3))               # sai: hàm trả về None

res = sai_chi_in(employees)
print("giá trị trả về:", res)
print("đúng:", type(selectFirstRows(employees)).__name__)

   employee_id     name   department  salary
0            3      Bob   Operations   48675
1           90    Alice        Sales   11096
2            9  Tatiana  Engineering   33805
giá trị trả về: None
đúng: DataFrame


**Vì sao sai:** đề yêu cầu **trả về** DataFrame. `print` chỉ hiển thị ra màn hình và hàm trả `None`, nên bộ chấm không có gì để so sánh.

### Lỗi 5: `loc` khi index có nhãn trùng

In [28]:
dup = pd.DataFrame({"v": [10, 20, 30, 40]}, index=[0, 0, 1, 2])
print(dup)
by_label = dup.loc[dup.index[:3]]          # nhãn [0, 0, 1]: mỗi nhãn 0 chọn ra CẢ HAI hàng nhãn 0
print(len(by_label), "hàng với loc theo nhãn (mong đợi 3)")
print(len(dup.iloc[:3]), "hàng với iloc (đúng 3)")
print(len(dup.head(3)), "hàng với head(3) (đúng 3)")
assert len(dup.head(3)) == 3 and len(by_label) != 3

    v
0  10
0  20
1  30
2  40
5 hàng với loc theo nhãn (mong đợi 3)
3 hàng với iloc (đúng 3)
3 hàng với head(3) (đúng 3)


**Vì sao nguy hiểm:** `loc` với danh sách nhãn trùng có thể **lặp** hàng. Khi chỉ muốn "n hàng đầu theo vị trí", dùng `head(n)` / `iloc[:n]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hiển thị / lấy **vài hàng đầu**" → `df.head(n)`. "Vài hàng cuối" → `df.tail(n)`.
- Nhớ: `head(n)` theo **vị trí**; `loc` theo **nhãn** (cận cuối được gồm); `iloc` theo **vị trí** (cận cuối không gồm).
- `head()` không có đối số lấy **5** hàng, không phải 3.
- Kết quả luôn **trả về** (return), không chỉ in.
- Khi không chắc kết quả, in `result.shape` và `result.index.tolist()`.
- Khi khám phá dữ liệu thật, `df.head()` là lệnh đầu tiên bạn nên gõ để "nhìn thử" bảng.

## 15. Biến thể của bài

1. **3 hàng cuối:** `df.tail(3)`.
2. **Hàng ở giữa:** `df.iloc[2:5]`.
3. **`n` hàng có lương cao nhất:** `df.nlargest(n, "salary")`.
4. **Lấy hàng đầu của từng nhóm:** `df.groupby("department").head(1)`.
5. **Lấy mẫu ngẫu nhiên có seed:** `df.sample(3, random_state=0)`.
6. **Đánh lại nhãn sau khi lấy:** `df.head(3).reset_index(drop=True)`.
7. Liên quan LeetCode 2880 *Select Data* (chọn theo điều kiện) và 2878 *Get the Size of a DataFrame*.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp thuần | `break` sau 3 hàng, dựng lại bảng | `O(k·c)` | `O(k·c)` | hiểu bản chất; không nên dùng thật |
| 2. `head(3)` | hàm có sẵn | `O(k·c)` | `O(k·c)` | **cách nộp bài**, rõ nhất |
| 3. `iloc[:3]` | cắt theo vị trí | `O(k·c)` | `O(k·c)` | muốn nhấn mạnh "theo vị trí" |
| 4. `df[:3]` | cú pháp tắt | `O(k·c)` | `O(k·c)` | tiện, nhưng dễ nhầm với chọn cột |
| 5. `loc[index[:3]]` / `take` | chọn theo nhãn / chỉ số | `O(k·c)` | `O(k·c)` | cần cẩn thận khi nhãn trùng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán số hàng của: `employees.head()`, `employees.head(0)`, `employees.head(-4)`, `employees.iloc[2:5]`, `employees.loc[1:3]` (bảng ví dụ 6 hàng, nhãn `0..5`).

**Bài 2 (tính tay).** Với `employees` (6 hàng) hãy liệt kê **tên** các nhân viên trong `employees.head(3)`, `employees.tail(2)` và `employees.iloc[1:4]`.

**Bài 3 (điền TODO).** Hoàn thành `last_rows_todo(df, n)` trả về `n` hàng cuối bằng cắt lát.

In [29]:
def last_rows_todo(df, n):
    # TODO 1: nếu n <= 0 trả về df.iloc[:0] (bảng rỗng cùng cột)
    # TODO 2: trả về df.iloc[-n:]   (n hàng cuối; nếu n lớn hơn số hàng thì lấy hết)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `select_first_rows_iloc(employees)` bằng `iloc` mà không nhìn code mẫu. Kiểm tra với ví dụ của đề và bảng đã sắp xếp theo `salary`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `top_paid(employees, n)` trả về `n` nhân viên có `salary` cao nhất (giữ nhãn gốc). (b) Viết `first_per_department(employees)` trả về **hàng đầu tiên** của mỗi `department` theo thứ tự xuất hiện trong bảng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `5`, `0`, `2` (bỏ 4 hàng cuối còn 2), `3` (vị trí 2,3,4), `3` (nhãn 1, 2, 3 vì `loc` gồm cận cuối). Bài 2: dựa vào bảng ở mục 2.
- **Mức 2 (vừa):** Bài 2: `head(3)` = Bob, Alice, Tatiana; `tail(2)` = Jonathan, Khaled; `iloc[1:4]` = Alice, Tatiana, Annabelle. Bài 3: nhớ `iloc[-n:]` với `n = 0` lấy **tất cả** (vì `-0 = 0`), nên cần nhánh riêng `n <= 0`.
- **Mức 3 (sâu):** Bài 5(a): `employees.nlargest(n, "salary")`; hoặc `sort_values("salary", ascending=False).head(n)` (hai cách có thể khác thứ tự khi lương trùng). Bài 5(b): `employees.groupby("department").head(1)`; hàng giữ nguyên nhãn gốc và thứ tự xuất hiện.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert len(employees.head()) == 5
assert len(employees.head(0)) == 0
assert len(employees.head(-4)) == 2
assert len(employees.iloc[2:5]) == 3
assert len(employees.loc[1:3]) == 3

# Bài 2
assert employees.head(3)["name"].tolist() == ["Bob", "Alice", "Tatiana"]
assert employees.tail(2)["name"].tolist() == ["Jonathan", "Khaled"]
assert employees.iloc[1:4]["name"].tolist() == ["Alice", "Tatiana", "Annabelle"]

# Bài 3
def last_rows_todo_ans(df, n):
    if n <= 0:
        return df.iloc[:0]
    return df.iloc[-n:]

pd.testing.assert_frame_equal(last_rows_todo_ans(employees, 2), employees.tail(2))
pd.testing.assert_frame_equal(last_rows_todo_ans(employees, 100), employees)
assert len(last_rows_todo_ans(employees, 0)) == 0

# Bài 4
def select_first_rows_iloc(employees):
    return employees.iloc[:3]

pd.testing.assert_frame_equal(select_first_rows_iloc(employees), expected_example)
pd.testing.assert_frame_equal(select_first_rows_iloc(shuffled), shuffled.head(3))

# Bài 5(a)
def top_paid(employees, n):
    return employees.nlargest(n, "salary")

assert top_paid(employees, 2)["name"].tolist() == ["Khaled", "Bob"]

# Bài 5(b)
def first_per_department(employees):
    return employees.groupby("department").head(1)

fp = first_per_department(employees)
assert fp["name"].tolist() == ["Bob", "Alice", "Tatiana", "Annabelle"]
assert fp.index.tolist() == [0, 1, 2, 3]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lấy `n` hàng đầu theo vị trí)

```python
import pandas as pd

def selectFirstRows(employees: pd.DataFrame) -> pd.DataFrame:
    return employees.head(3)
```

**Checklist**

- [ ] Dùng `head(3)` (hoặc `iloc[:3]`), không phải `head(-3)` hay `loc[:3]`?
- [ ] **Trả về** DataFrame (`return`), không chỉ `print`?
- [ ] Hiểu: theo **vị trí** (`head`, `iloc`) khác theo **nhãn** (`loc`)?
- [ ] Không đổi cột / không đổi thứ tự hàng (không `sort_values` trước)?
- [ ] Đã thử bảng có < 3 hàng, bảng đã sắp xếp (nhãn lộn xộn)?
- [ ] Skeleton đúng `def selectFirstRows(employees: pd.DataFrame) -> pd.DataFrame` (đối chiếu lại với đề gốc nếu lệch)?